# 🖼️ Vision — image input with pi.dev

Notebook 06 of the pi.dev series. So far every request has been **text → text**.
pi-ai messages are multimodal: a single user message can carry **text *and* image**
blocks. Here we send an image alongside a question and let the model describe it.

> **⚠️ Requires a vision-capable deployment.** Image input only works if your
> Azure deployment is a vision model (e.g. a **GPT-4o / GPT-4.1**-class deployment).
> If your `AZURE_PI_TEST_DEPLOYMENT` is text-only (like `cohere-command-a`), the
> request will error — that's expected, and the notebook still shows you the
> **correct request shape** so you can point it at a vision deployment later.

> **Kernel:** Deno (pi.dev is ESM-only TypeScript; the Deno kernel runs it natively).

## 1. Load env & register Azure OpenAI

Same first step as every notebook: `loadEnvUp()` pulls the `AZURE_PI_TEST_*` keys
from a `.env` outside the repo.

In [ ]:
import { loadEnvUp } from "playground/env";

const env = await loadEnvUp();
console.log(`\n${env.files.length} .env file(s) found; ${env.loaded.length} variable(s) loaded.`);

### Register with **image** input enabled

`registerAzure()` defaults each model to `input: ["text"]`. To send images we must
declare `input: ["text", "image"]` — the helper takes a `modelOverrides` object that
is merged into every registered deployment, so we flip it on here.

This only changes what pi-ai is *willing to send*; the deployment behind
`AZURE_PI_TEST_DEPLOYMENT` still has to actually support vision.

In [ ]:
import { registerAzure } from "playground/azure";

const { models, model, modelId } = registerAzure({
  modelOverrides: { input: ["text", "image"] },
});
console.log(`Model ${modelId} accepts input types: ${model.input.join(", ")}`);

## 2. The multimodal message shape

A `UserMessage.content` can be either a plain string (what we used before) **or an
array of content blocks**. For vision we mix two block types:

```ts
{ type: "text",  text: "Describe this image." }        // TextContent
{ type: "image", data: "<base64>", mimeType: "image/png" }  // ImageContent
```

The `data` field is the **base64-encoded bytes** of the image (no `data:` URI
prefix), and `mimeType` tells the model how to decode them.

## 3. A tiny embedded test image

To keep the notebook self-contained (no network, no external files) we embed a
small **64×64 PNG: a solid red circle on a white background**, base64-encoded
below. A vision model should describe it as roughly “a red circle/dot on white.”

In [ ]:
// A 64x64 PNG: red circle on white, embedded as base64 (no data: prefix).
const imageBase64 =
  "iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAABF0lEQVR4nOWZy3LEIBADRZf//5e1h73tJhXzMKDQV9sjicEuylNsKxkUDgoHhYPCQeGgcFA4KBwUDgoHhXM9UrWUXy+NPnpdk3x/3zMoyTXP+o+P2KsDNFgfGoMB8v2UsiLAKPd91djCfUdNdnHfWpmN3DfVZy/39SrxRwm2W/5KrXM6UCYuf43iOR3IDlCm75/buod0YGNQOCgcFA4KB4WDjgjgRYNA/617SAf+QwBP30W+pXhOBzS3Cb6rdVQHNKsJrlCp78DTGVxXv2kLPZfB1ZVb34EnMrilZsdLPDaDG6v1fYVGZbDXjZje2s2/Lbx8RtYcY68p5Yen1DnxikMHCgeFg8JB4aBwUDgoHBQOCofVBnp5AdbYPYs/EqvzAAAAAElFTkSuQmCC";
const mimeType = "image/png";
console.log(`Embedded image: ${imageBase64.length} base64 chars (${mimeType}).`);

## 4. Send the image and ask for a description

We build a `Context` whose single user message carries both the text prompt and the
image block, then call `completeSimple`. We wrap it in `try/catch`: on a text-only
deployment the provider rejects the image and we report that cleanly instead of
crashing.

In [ ]:
import { type Context } from "@earendil-works/pi-ai";

const visionContext: Context = {
  systemPrompt: "You are a concise visual assistant.",
  messages: [
    {
      role: "user",
      content: [
        { type: "text", text: "Describe this image in one sentence." },
        { type: "image", data: imageBase64, mimeType },
      ],
      timestamp: Date.now(),
    },
  ],
};

try {
  const visionReply = await models.completeSimple(model, visionContext);
  const texts = visionReply.content.filter((b) => b.type === "text");
  if (texts.length) {
    for (const b of texts) console.log("\uD83E\uDD16", (b as { text: string }).text);
  } else {
    console.log("\u26A0\uFE0F  No text returned. stopReason:", visionReply.stopReason);
  }
  if (visionReply.errorMessage) console.log("errorMessage:", visionReply.errorMessage);
  console.log(
    `\ntokens: ${visionReply.usage.input} in / ${visionReply.usage.output} out` +
      `  |  cost: $${visionReply.usage.cost.total.toFixed(6)}`,
  );
} catch (err) {
  console.log("\uD83D\uDCA5 Request failed \u2014 is your deployment vision-capable?");
  console.log(String(err).split("\n")[0]);
  console.log(
    "\nSet AZURE_PI_TEST_DEPLOYMENT to a GPT-4o / GPT-4.1-class deployment and re-run.",
  );
}

## 5. Loading a real image file (reference)

The embedded base64 above is just for a self-contained demo. In practice you'll read
an image from disk. In Deno that looks like:

```ts
const bytes = await Deno.readFile("pic.png");
const b64 = btoa(String.fromCharCode(...bytes));
// then: { type: "image", data: b64, mimeType: "image/png" }
```

For large images prefer a chunked/`FileReader`-style base64 conversion to avoid
blowing the call stack with the spread operator.

## ✅ What just happened

1. We enabled image input by registering the model with `input: ["text", "image"]`
   via `registerAzure({ modelOverrides })`.
2. We built a `UserMessage` whose `content` is an **array** mixing a `text` block and
   an `image` block (base64 `data` + `mimeType`).
3. `completeSimple` sent both to the model; a vision deployment describes the image,
   while a text-only deployment returns an error we caught and explained.

The request **shape** is identical regardless of provider — that's the point of
pi-ai's unified message model.

**Next:** notebook **07 — Reasoning / thinking** (capturing the model's thinking
blocks and controlling reasoning effort).